In [ ]:
include( "../geom.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/spd-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim

home, base = expfolder( "system" )

colormap = cgrad( :ice )

In [ ]:
# Hyper parameters.
N = 1024;  s = 0.251

# Create folder and extract critical noise.
folderlist = [createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) ) for ρ ∈ ρlist]

# Extract critical noise values.
ηclist = first.( readdlm.( folderlist.*"../order-0.5.txt" ) );

In [ ]:
# Initialize data variables.
θlist = Matrix{defFloat}( undef, Nρ, 3 )
μℓdata = Vector{Vector{defFloat}}( undef, Nρ )
σℓdata = Vector{Vector{defFloat}}( undef, Nρ )
μAdata = Vector{Vector{defFloat}}( undef, Nρ )
σAdata = Vector{Vector{defFloat}}( undef, Nρ )

# Import cluster dynamical data.
for i ∈ 1:Nρ
    # Unpack the coefficients.
    θlist[i,:] = readdlm( folderlist[i]*"area-perimeter-scaling.txt" )

    # Unpack the geometry of each cluster size.
    ℓdata = readdlm( folderlist[i]*"mean-convex-hull-perimeter.txt" )
    Adata = readdlm( folderlist[i]*"mean-convex-hull-area.txt" )

    # Unpack the mean and standard deviation separately.
    μℓdata[i] = ℓdata[:,1];  σℓdata[i] = ℓdata[:,2]
    μAdata[i] = Adata[:,1];  σAdata[i] = Adata[:,2]
end

In [ ]:
# Choose focal parameter combination.
i = 15
println( ρlist[i] )

# Unpack data for plotting.
ηc = ηclist[i];  θ = θlist[i,1:2]
μℓlist = μℓdata[i];  σℓlist = σℓdata[i]
μAlist = μAdata[i];  σAlist = σAdata[i]

# Unpack the scaling parameters across indeces.
βlist = θlist[:,1];  ℓ0list = θlist[:,2]

# Cluster sizes to consider in plot.
jlist = 1:N
ιlist = 2:length( μℓlist )

# Plot dimensions.
ℓmin = floor( log10( minimum( μℓlist ) ) )
ℓmax = ceil(  log10( maximum( μℓlist ) ) );

In [ ]:
# Plot the distribution of ℓ across i.
plt = plot(; size=(375,250), legend=:topleft )
plot!( plt; left_margin=0pt, right_margin=10pt, bottom_margin=0pt )

# Plot mean and fit.
scatter!( plt, ιlist, μℓlist; yerror=σℓlist,
    color=:gray, markersize=3.0, lw=1.5, fillcolor=:gray, alpha=1, label="" )
plot!( plt, jlist, polyloglaw.( jlist; β=θ ); color=:cornflowerblue, lw=2, label=L"ℓ_0 (\log(i+1))^β" )

xticks = 2.0.^(0:2:log2( N ))
plot!( plt; xlims=(1,N), xticks=(xticks, string.( round.( defInt, xticks ) )), xscale=:log10 )
plot!( plt; ylims=10.0.^[ℓmin,ℓmax], yscale=:log10 )
plot!( plt; xlabel="cluster size, "*L"i", ylabel="convex hull perimeter, "*L"ℓ_i" )

# Create inset for perimeter.
plot!( plt; inset=(1, bbox( 0.55, 0.45, 0.325, 0.35 )) )
pltℓ0 = twinx( plt[2] )

scatter!( plt[2], ρlist, βlist; color=:gray, label="" )
scatter!( plt[3], ρlist, ℓ0list; color=:cornflowerblue, label="" )

xlims = 10.0.^[-2,0]
ylims = [2,2.6]
plot!( plt[2]; xlims=xlims, xticks=xlims, xscale=:log10 )
plot!( plt[2]; ylims=ylims, yticks=ylims )

ylims = [0.6,0.75]
plot!( plt[3]; xlims=xlims, xticks=[], xscale=:log10 )
plot!( plt[3]; ylims=ylims, yticks=ylims )
plot!( plt[3]; tickfontsize=9, tickfontcolor=:royalblue )

plot!( plt[1]; framestyle=:box )
plot!( plt[2]; framestyle=:box, tickfontsize=9 )

annotate!( plt[2], 10^(-1.0), 1.875, text( latexstring( "ρ" ), 9, :center ) )
annotate!( plt[2], 10^(-2.3), 2.3, text( latexstring( "β" ), 9, :center; rotation=90 ) )
annotate!( plt[2], 10^(0.35), 2.3, text( latexstring( "ℓ_0" ), 9, :center; rotation=270, color=:royalblue ) )

# saveplot( plt, figurefolder*"geometry-area-perimeter_N-$(N).png"; dpi=600 );

In [ ]:
# Plot the distribution of ℓ across i.
plt = plot(; layout=@layout( [a{0.6h};[b{0.5w} c]] ), size=(400,475), legend=:topleft )
plot!( plt; right_margin=10pt )

# Plot mean and fit.
scatter!( plt[1], ιlist, μℓlist; yerror=σℓlist,
    color=:gray, markersize=3.0, lw=1.5, fillcolor=:gray, alpha=1, label="" )
plot!( plt[1], jlist, polyloglaw.( jlist; β=θ ); color=:cornflowerblue, lw=2, label=L"ℓ_0 (\log(i+1))^β" )

xticks = 2.0.^(0:2:log2( N ))
plot!( plt[1]; xlims=(1,N), xticks=(xticks, string.( round.( defInt, xticks ) )), xscale=:log10 )
plot!( plt[1]; ylims=10.0.^[ℓmin,ℓmax], yscale=:log10 )
plot!( plt[1]; xlabel="cluster size, "*L"i", ylabel="convex hull perimeter, "*L"ℓ_i" )
plot!( plt[1]; framestyle=:box )

# Create inset for perimeter.
plot!( plt; inset=(1, bbox( 0.55, 0.45, 0.325, 0.35 )) )
pltℓ0 = twinx( plt[4] )

scatter!( plt[4], ρlist, βlist; color=:gray, label="" )
scatter!( plt[5], ρlist, ℓ0list; color=:cornflowerblue, label="" )

xlims = 10.0.^[-2,0]
ylims = [2,2.6]
plot!( plt[4]; xlims=xlims, xticks=xlims, xscale=:log10 )
plot!( plt[4]; ylims=ylims, yticks=ylims )

ylims = [0.6,0.75]
plot!( plt[5]; xlims=xlims, xticks=[], xscale=:log10 )
plot!( plt[5]; ylims=ylims, yticks=ylims )
plot!( plt[5]; tickfontsize=9, tickfontcolor=:royalblue )

plot!( plt[4]; framestyle=:box, tickfontsize=9 )

annotate!( plt[4], 10^(-1.0), 1.875, text( latexstring( "ρ" ), 9, :center ) )
annotate!( plt[4], 10^(-2.3), 2.3, text( latexstring( "β" ), 9, :center; rotation=90 ) )
annotate!( plt[4], 10^(0.35), 2.3, text( latexstring( "ℓ_0" ), 9, :center; rotation=270, color=:royalblue ) )

# Plot the change in exponent for varying density and speed.
heatmap!( plt[2], ρlist, slist, first.( γdata ); color=colormap, colorbar=false )

plot!( plt[2]; xlims=ρlist[[1,Nρ]], xscale=:log10 )
plot!( plt[2]; ylims=slist[[1,Ns]], yscale=:log10 )

# Plot the change in scaling constant for changing density and speed.
heatmap!( plt[3], ρlist, slist, 10.0.^last.( γdata ); color=colormap, colorbar=false )

plot!( plt[3]; xlims=ρlist[[1,Nρ]], xscale=:log10 )
plot!( plt[3]; ylims=slist[[1,Ns]], yscale=:log10 )

# saveplot( plt, figurefolder*"geometry-area-perimeter_N-$(N).png"; dpi=600 );